# Validation Experiment — nb01
_Train → per-layer trajectory similarity → z-space proxy validation_

---

| Step | Estimated time (Colab T4) |
|------|--------------------------|
| Train Option A | ~45 min |
| Train Option B | ~60 min |
| Per-layer trajectory similarity (1000 pairs) | ~2 min |
| z-space proxy validation + plots | ~5 min |
| Augmentation invariance | ~3 min |
| UMAP visualization | ~2 min |
| **Total** | **~2–2.5 hours** |

---
## 0. Setup

In [ ]:
import os
import sys

# ── Colab detection ────────────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'
    REPO_DIR = '/content/ctls'

    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
        os.system(f'pip install -r {REPO_DIR}/requirements.txt -q')

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive/ctls'
    os.makedirs(f'{DRIVE_BASE}/data',        exist_ok=True)
    os.makedirs(f'{DRIVE_BASE}/experiments', exist_ok=True)

    if not os.path.exists(f'{REPO_DIR}/data'):
        os.symlink(f'{DRIVE_BASE}/data',        f'{REPO_DIR}/data')
    if not os.path.exists(f'{REPO_DIR}/experiments'):
        os.symlink(f'{DRIVE_BASE}/experiments', f'{REPO_DIR}/experiments')

    ROOT = REPO_DIR
else:
    # Notebook lives at notebooks/validation/ — go up two levels
    ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f'ROOT: {ROOT}')

In [ ]:
import yaml
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr

from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder
from data.cifar import get_standard_loaders
from evaluation.circuit_viz import CircuitVisualizer
from evaluation.embedding_compare import EmbeddingComparator
from evaluation.activation_patching import ActivationPatcher, sample_pairs

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Model loading helper ───────────────────────────────────────────────────────
def load_unified_model(config_path, checkpoint_path, device):
    """Load a model trained by UnifiedTrainer from a config + checkpoint path."""
    with open(config_path) as f:
        cfg = yaml.safe_load(f)

    mcfg = cfg['model']
    ecfg = mcfg['meta_encoder']
    tcfg = cfg['training']

    soft_mask = SoftMask(init_temperature=tcfg['temperature']['init'])
    backbone = CTLSBackbone(
        arch=mcfg['arch'],
        num_classes=mcfg['num_classes'],
        soft_mask=soft_mask,
        pretrained=mcfg.get('pretrained', False),
    ).to(device)

    meta_encoder = MetaEncoder(
        layer_dims=backbone.layer_dims,
        embedding_dim=ecfg.get('embedding_dim', 64),
        encoder_type=ecfg.get('encoder_type', 'weighted_sum'),
        projection_dim=ecfg.get('projection_dim', 128),
    ).to(device)

    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    backbone.load_state_dict(ckpt['backbone_state'])
    meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
    backbone.eval()
    meta_encoder.eval()

    print(f"Loaded {config_path} | epoch={ckpt['epoch']} | val_acc={ckpt['val_acc']:.4f}")
    return backbone, meta_encoder, cfg

---
## 1. Train

> Skip if checkpoints already exist — cells below detect and skip automatically.

In [ ]:
# ── 1a. Train Option A (weighted_sum, d=128) ───────────────────────────────
# Estimated time: ~45 min on T4

CKPT_A = 'experiments/unified_a/best.pt'

if os.path.exists(CKPT_A):
    print(f'Checkpoint found at {CKPT_A} — skipping training. Delete to retrain.')
else:
    from training.unified_trainer import UnifiedTrainer

    with open('configs/unified_a.yaml') as f:
        config_a = yaml.safe_load(f)

    trainer_a = UnifiedTrainer(config_a)
    trainer_a.train()

In [ ]:
# ── 1b. Train Option B (transformer_cls, d=128) ────────────────────────────
# Estimated time: ~60 min on T4 (transformer is slower to train)

CKPT_B = 'experiments/unified_b/best.pt'

if os.path.exists(CKPT_B):
    print(f'Checkpoint found at {CKPT_B} — skipping training. Delete to retrain.')
else:
    from training.unified_trainer import UnifiedTrainer

    with open('configs/unified_b.yaml') as f:
        config_b = yaml.safe_load(f)

    trainer_b = UnifiedTrainer(config_b)
    trainer_b.train()

In [ ]:
# ── 1c. Load both models ───────────────────────────────────────────────────
backbone_a, meta_encoder_a, cfg_a = load_unified_model(
    'configs/unified_a.yaml', CKPT_A, DEVICE
)
backbone_b, meta_encoder_b, cfg_b = load_unified_model(
    'configs/unified_b.yaml', CKPT_B, DEVICE
)

# Standard val loader for evaluation
_, val_loader = get_standard_loaders(
    data_dir='data/cifar10',
    batch_size=256,
    num_workers=4,
    augment=False,
    download=True,
)

In [ ]:
# ── 1d. Sanity-check metrics ───────────────────────────────────────────────
# Replicate the key metrics from Stage 2 to verify the unified objective
# still achieves circuit organisation (Step 1 sanity check from roadmap).

results = {}

for label, backbone, meta_encoder in [
    ('Option A (weighted_sum)', backbone_a, meta_encoder_a),
    ('Option B (transformer_cls)', backbone_b, meta_encoder_b),
]:
    viz = CircuitVisualizer(backbone, meta_encoder, val_loader, DEVICE)
    cmp = EmbeddingComparator(backbone, meta_encoder, DEVICE)

    circuit_sil = viz.cluster_separation_score(max_samples=5000)
    intraclass_rho_dict = cmp.intraclass_distance_rank(val_loader, n_samples=2000)
    noise_result = cmp.compare_clean_vs_degraded(val_loader, noise_std=0.3, n_samples=1000)

    mean_rho = float(np.mean([v['spearman_rho'] for v in intraclass_rho_dict.values()]))

    results[label] = {
        'circuit_silhouette': circuit_sil['silhouette_circuit'],
        'output_silhouette':  circuit_sil['silhouette_output'],
        'intraclass_rho':     mean_rho,
        'noise_ratio_0.3':    noise_result['ratio_circuit_over_output'],
    }
    print(f"\n{label}")
    for k, v in results[label].items():
        print(f"  {k}: {v:.4f}")

print("\n" + "─"*60)
print("Sanity check thresholds:")
print("  circuit_silhouette > 0.6  → unified objective organising circuits")
print("  output_silhouette  unchanged vs. baseline (0.81) → backbone intact")

---
## 2. Per-Layer Trajectory Similarity — Ground Truth

For each pair (x_a, x_b), compute cosine similarity between activations at each layer independently:

```
sim_l(x_a, x_b) = cos( h_l(x_a), h_l(x_b) )   for l = 1..L
```

No cascade effect, no interventions — each layer measured in isolation.

**What to look for:** Same-class pairs should be more similar than diff-class pairs at every layer, with the gap widening at deeper layers. This continuous per-layer profile is the ground truth used in Section 3.

In [ ]:
# ── 2a. Build pair set ─────────────────────────────────────────────────────
# 500 same-class + 500 different-class pairs from the val set.
# Saved to disk so the run can be resumed across Colab sessions.

import torchvision
import torchvision.transforms as T

os.makedirs('experiments/validation', exist_ok=True)
PAIRS_SAVE = 'experiments/validation/pairs.pt'

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)
val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])
val_dataset = torchvision.datasets.CIFAR10(
    root='data/cifar10', train=False, transform=val_transform, download=True
)

if os.path.exists(PAIRS_SAVE):
    saved = torch.load(PAIRS_SAVE, weights_only=False)
    pairs       = saved['pairs']
    pair_labels = saved['pair_labels']
    pair_classes = saved['pair_classes']
    print(f'Loaded {len(pairs)} pairs from {PAIRS_SAVE}')
else:
    pairs, pair_labels, pair_classes = sample_pairs(
        val_dataset, n_same=500, n_diff=500, seed=42
    )
    torch.save(
        {'pairs': pairs, 'pair_labels': pair_labels, 'pair_classes': pair_classes},
        PAIRS_SAVE
    )
    print(f'Sampled {len(pairs)} pairs and saved to {PAIRS_SAVE}')

print(f'  Same-class pairs:  {pair_labels.sum()}')
print(f'  Diff-class pairs:  {(1 - pair_labels).sum()}')

In [ ]:
# ── 2b. Compute per-layer cosine similarity profiles ──────────────────────
# sim_l(x_a, x_b) = cos(h_l(x_a), h_l(x_b)) for each layer l.
# Two forward passes per pair — no hooks, no interventions. Runs in ~2 min.

TRAJ_SIM_SAVE = 'experiments/validation/trajectory_similarities.pt'

@torch.no_grad()
def compute_trajectory_similarities(backbone, pairs, dataset, device):
    """Returns [N, L] array of per-layer cosine similarities for all pairs."""
    backbone.eval()
    all_sims = []
    for idx_a, idx_b in pairs:
        x_a = dataset[idx_a][0].unsqueeze(0).to(device)
        x_b = dataset[idx_b][0].unsqueeze(0).to(device)
        _, traj_a = backbone(x_a)
        _, traj_b = backbone(x_b)
        layer_sims = [
            F.cosine_similarity(h_a, h_b).item()
            for h_a, h_b in zip(traj_a, traj_b)
        ]
        all_sims.append(layer_sims)
    return np.array(all_sims, dtype=np.float32)  # [N, L]

if os.path.exists(TRAJ_SIM_SAVE):
    saved_ts = torch.load(TRAJ_SIM_SAVE, weights_only=False)
    per_layer_sims_a = saved_ts['per_layer_sims_a']
    per_layer_sims_b = saved_ts['per_layer_sims_b']
    print(f'Loaded trajectory similarities from {TRAJ_SIM_SAVE}')
else:
    print('Computing per-layer trajectory similarities for Option A...')
    per_layer_sims_a = compute_trajectory_similarities(backbone_a, pairs, val_dataset, DEVICE)
    print('Computing per-layer trajectory similarities for Option B...')
    per_layer_sims_b = compute_trajectory_similarities(backbone_b, pairs, val_dataset, DEVICE)
    torch.save({'per_layer_sims_a': per_layer_sims_a, 'per_layer_sims_b': per_layer_sims_b},
               TRAJ_SIM_SAVE)
    print(f'Saved to {TRAJ_SIM_SAVE}')

same_mask = pair_labels == 1
diff_mask  = pair_labels == 0

# Scalar ground truth: mean cosine similarity across all layers
traj_sim_a = per_layer_sims_a.mean(axis=1)  # [N]
traj_sim_b = per_layer_sims_b.mean(axis=1)  # [N]

for lbl, ts in [('Option A', traj_sim_a), ('Option B', traj_sim_b)]:
    sep = ts[same_mask].mean() - ts[diff_mask].mean()
    print(f'{lbl}: same={ts[same_mask].mean():.4f}  diff={ts[diff_mask].mean():.4f}  sep={sep:.4f}')

In [ ]:
# ── 2c. Per-layer similarity profiles ──────────────────────────────────────
# Mean cosine similarity at each layer, split by pair type. Shaded band = ±1 std.

L      = per_layer_sims_a.shape[1]
layers = np.arange(1, L + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, pls, title in [
    (axes[0], per_layer_sims_a, 'Option A (weighted_sum)'),
    (axes[1], per_layer_sims_b, 'Option B (transformer_cls)'),
]:
    for mask, color, label in [
        (same_mask, 'steelblue', 'Same-class'),
        (diff_mask,  'tomato',   'Diff-class'),
    ]:
        m = pls[mask].mean(axis=0)
        s = pls[mask].std(axis=0)
        ax.plot(layers, m, 'o-', color=color, label=label)
        ax.fill_between(layers, m - s, m + s, alpha=0.15, color=color)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Mean cosine similarity')
    ax.set_title(f'{title}\nPer-layer trajectory similarity')
    ax.set_xticks(layers)
    ax.set_ylim(-0.1, 1.05)
    ax.legend()

plt.tight_layout()
plt.savefig('experiments/validation/per_layer_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2d. Trajectory similarity distributions ────────────────────────────────
# Same-class and diff-class distributions of mean per-layer similarity.
# Should show overlapping but separated distributions (unlike the old binary CircuitSim).

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, ts, title in [
    (axes[0], traj_sim_a, 'Option A (weighted_sum)'),
    (axes[1], traj_sim_b, 'Option B (transformer_cls)'),
]:
    ax.hist(ts[same_mask], bins=30, alpha=0.6, label='Same-class', color='steelblue', density=True)
    ax.hist(ts[diff_mask],  bins=30, alpha=0.6, label='Diff-class',  color='tomato',    density=True)
    ax.set_xlabel('Mean trajectory cosine similarity')
    ax.set_ylabel('Density')
    ax.set_title(title)
    ax.legend()

plt.tight_layout()
plt.savefig('experiments/validation/traj_sim_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2e. Per-layer same/diff gap ────────────────────────────────────────────
# How class-discriminative is each layer? Gap = mean(same) - mean(diff).

print(f'{"Layer":<8} {"Gap A":>10} {"Gap B":>10}')
print('─' * 30)
for l in range(per_layer_sims_a.shape[1]):
    gap_a = per_layer_sims_a[same_mask, l].mean() - per_layer_sims_a[diff_mask, l].mean()
    gap_b = per_layer_sims_b[same_mask, l].mean() - per_layer_sims_b[diff_mask, l].mean()
    print(f'{l+1:<8} {gap_a:>10.4f} {gap_b:>10.4f}')

---
## 3. z-Space Proxy Validation

Three tests, in order of importance:

1. **Primary ρ** — Spearman ρ between z-space cosine similarity and mean trajectory similarity (Section 2 ground truth). Does z track the trajectory?
2. **Per-layer ρ** — For each layer l, ρ(z-sim, sim_l). Which layers does z most strongly reflect?
3. **Baseline comparison** — z-sim vs. h₈ cosine sim vs. h₁ cosine sim, evaluated on same/diff-class discrimination. Does the trajectory encoder add over the simplest alternatives?

| ρ | Result |
|---|--------|
| > 0.5 | Proxy validated — z tracks trajectory structure |
| 0.3–0.5 | Weak proxy |
| < 0.3 | Proxy invalid — revisit architecture |

In [ ]:
# ── 3a. Compute z-space cosine similarities ───────────────────────────────
@torch.no_grad()
def compute_z_similarities(backbone, meta_encoder, pairs, dataset, device):
    """For each (idx_a, idx_b) pair, return cosine similarity of z vectors."""
    backbone.eval(); meta_encoder.eval()
    z_sims = []
    for idx_a, idx_b in pairs:
        x_a = dataset[idx_a][0].unsqueeze(0).to(device)
        x_b = dataset[idx_b][0].unsqueeze(0).to(device)
        _, traj_a = backbone(x_a)
        _, traj_b = backbone(x_b)
        z_a = meta_encoder(traj_a)
        z_b = meta_encoder(traj_b)
        z_sims.append(F.cosine_similarity(z_a, z_b).item())
    return np.array(z_sims, dtype=np.float32)

print('Computing z-space similarities for Option A...')
z_sims_a = compute_z_similarities(backbone_a, meta_encoder_a, pairs, val_dataset, DEVICE)
print(f'  same={z_sims_a[same_mask].mean():.4f}  diff={z_sims_a[diff_mask].mean():.4f}')

print('Computing z-space similarities for Option B...')
z_sims_b = compute_z_similarities(backbone_b, meta_encoder_b, pairs, val_dataset, DEVICE)
print(f'  same={z_sims_b[same_mask].mean():.4f}  diff={z_sims_b[diff_mask].mean():.4f}')

In [ ]:
# ── 3b. Primary Spearman ρ — z-sim vs. mean trajectory similarity ──────────
rho_a, pval_a = spearmanr(z_sims_a, traj_sim_a)
rho_b, pval_b = spearmanr(z_sims_b, traj_sim_b)

print('Spearman ρ: z-space cosine similarity vs. mean per-layer trajectory similarity')
print(f'  Option A (weighted_sum):    ρ = {rho_a:.4f}  p = {pval_a:.2e}')
print(f'  Option B (transformer_cls): ρ = {rho_b:.4f}  p = {pval_b:.2e}')

In [ ]:
# ── 3c. Scatter — z-sim vs. trajectory similarity ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, z_sims, ts, rho, pval, title in [
    (axes[0], z_sims_a, traj_sim_a, rho_a, pval_a, 'Option A (weighted_sum)'),
    (axes[1], z_sims_b, traj_sim_b, rho_b, pval_b, 'Option B (transformer_cls)'),
]:
    ax.scatter(ts[same_mask], z_sims[same_mask], c='steelblue', alpha=0.4, s=15, label='Same-class')
    ax.scatter(ts[diff_mask],  z_sims[diff_mask],  c='tomato',    alpha=0.4, s=15, label='Diff-class')
    ax.set_xlabel('Mean trajectory cosine similarity (ground truth)')
    ax.set_ylabel('z-space cosine similarity')
    ax.set_title(f'{title}\nSpearman ρ = {rho:.3f}  (p={pval:.2e})')
    ax.legend(markerscale=2)

plt.tight_layout()
plt.savefig('experiments/validation/proxy_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 3d. Per-layer ρ — which layers does z track most strongly? ─────────────
# For each layer l: Spearman ρ between z-sim and sim_l(x_a, x_b).
# Reveals whether z is dominated by early, late, or all layers.

L      = per_layer_sims_a.shape[1]
layers = np.arange(1, L + 1)

per_layer_rho_a = [spearmanr(z_sims_a, per_layer_sims_a[:, l])[0] for l in range(L)]
per_layer_rho_b = [spearmanr(z_sims_b, per_layer_sims_b[:, l])[0] for l in range(L)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(layers, per_layer_rho_a, 'o-',  color='steelblue',  label='Option A (weighted_sum)')
ax.plot(layers, per_layer_rho_b, 's--', color='darkorange', label='Option B (transformer_cls)')
ax.axhline(0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('Layer')
ax.set_ylabel('Spearman ρ  (z-sim vs. layer cosine sim)')
ax.set_title('Which layers does z most strongly reflect?')
ax.set_xticks(layers)
ax.legend()
plt.tight_layout()
plt.savefig('experiments/validation/per_layer_rho.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'{"Layer":<8} {"Option A ρ":>12} {"Option B ρ":>12}')
print('─' * 34)
for l, (r_a, r_b) in enumerate(zip(per_layer_rho_a, per_layer_rho_b), 1):
    print(f'{l:<8} {r_a:>12.3f} {r_b:>12.3f}')

In [ ]:
# ── 3e. Baseline comparison — does z add over single-layer proxies? ─────────
# Spearman ρ vs. pair_labels (1=same-class, 0=diff-class) for competing signals.
# If z > h₈: the full trajectory is contributing beyond the last layer alone.

h8_sims_a    = per_layer_sims_a[:, -1]        # last layer
h1_sims_a    = per_layer_sims_a[:,  0]        # first layer
h8_sims_b    = per_layer_sims_b[:, -1]
h1_sims_b    = per_layer_sims_b[:,  0]

print('Spearman ρ vs. pair label (1=same-class, 0=diff-class):')
print(f'{"Signal":<28} {"Option A":>10} {"Option B":>10}')
print('─' * 50)
for lbl, va, vb in [
    ('z-space cosine sim',     z_sims_a, z_sims_b),
    ('h₈ cosine sim (last)',   h8_sims_a, h8_sims_b),
    ('h₁ cosine sim (first)',  h1_sims_a, h1_sims_b),
]:
    r_a, _ = spearmanr(va, pair_labels)
    r_b, _ = spearmanr(vb, pair_labels)
    print(f'{lbl:<28} {r_a:>10.4f} {r_b:>10.4f}')
print()
print('If z > h₈: trajectory encoding adds information the last layer alone does not have.')
print('If z ≈ h₈: the last layer dominates; the multi-layer machinery needs justification.')

In [ ]:
# ── 3f. Bootstrap confidence intervals ────────────────────────────────────
def bootstrap_spearman_ci(x, y, n_bootstrap=2000, seed=42):
    rng = np.random.default_rng(seed)
    rhos = []
    for _ in range(n_bootstrap):
        idx = rng.integers(0, len(x), size=len(x))
        rho, _ = spearmanr(x[idx], y[idx])
        rhos.append(rho)
    return float(np.percentile(rhos, 2.5)), float(np.percentile(rhos, 97.5))

ci_a = bootstrap_spearman_ci(z_sims_a, traj_sim_a)
ci_b = bootstrap_spearman_ci(z_sims_b, traj_sim_b)

print('Spearman ρ with 95% bootstrap confidence intervals (n=2000 resamples):')
print(f'  Option A: ρ = {rho_a:.4f}  95% CI [{ci_a[0]:.4f}, {ci_a[1]:.4f}]')
print(f'  Option B: ρ = {rho_b:.4f}  95% CI [{ci_b[0]:.4f}, {ci_b[1]:.4f}]')
print()
print('A CI that excludes 0.0 confirms the correlation is not a sampling artifact.')

In [ ]:
# ── 3g. Validation summary ─────────────────────────────────────────────────
print('=' * 65)
print('VALIDATION SUMMARY')
print('=' * 65)
print()
print(f'Option A: ρ = {rho_a:.4f}  95% CI [{ci_a[0]:.4f}, {ci_a[1]:.4f}]')
print(f'Option B: ρ = {rho_b:.4f}  95% CI [{ci_b[0]:.4f}, {ci_b[1]:.4f}]')
print()

def interpret(rho, label):
    if rho > 0.5:
        return f'{label}: VALIDATED — z reliably tracks per-layer trajectory structure.'
    elif rho > 0.3:
        return f'{label}: WEAK — some signal, but not robust. Try Step 4 (extraction ablation).'
    else:
        return f'{label}: INVALID — z is not tracking trajectory similarity. Revisit architecture.'

print(interpret(rho_a, 'Option A'))
print(interpret(rho_b, 'Option B'))
print()
print('Silhouette scores (Section 1):')
for label, res in results.items():
    print(f'  {label}: circuit={res["circuit_silhouette"]:.4f}  output={res["output_silhouette"]:.4f}')
print('=' * 65)

---
## 4. Augmentation Invariance

If z tracks *how* the model processes an image — not just surface pixel statistics — it should assign similar vectors to different augmented views of the same image.

**Test:** compute within-image z-similarity (K augmented views) and compare against same-class and diff-class pair similarities from Section 3.

**Expected ordering:** within-image > same-class > diff-class.
If within-image ≈ same-class, z is stable under augmentation but cannot resolve individual instances. If within-image < same-class, z may be tracking the class label rather than instance-level circuit patterns.

In [ ]:
# ── 4a. Generate augmented views and compute z ────────────────────────────
import torchvision.transforms as T

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2470, 0.2435, 0.2616)

augment_transform = T.Compose([
    T.RandomResizedCrop(32, scale=(0.75, 1.0)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
])

# Raw PIL dataset — no pre-applied transform, so augment_transform applies fresh each call
raw_dataset = torchvision.datasets.CIFAR10(root='data/cifar10', train=False, download=True)

N_IMAGES = 200
K_VIEWS  = 5

rng_aug = np.random.default_rng(99)
aug_indices = rng_aug.choice(len(raw_dataset), size=N_IMAGES, replace=False)

@torch.no_grad()
def get_z_for_views(backbone, meta_encoder, indices, dataset, transform, K, device):
    """Returns [N, K, D] tensor — K augmented-view z vectors for each of N images."""
    backbone.eval(); meta_encoder.eval()
    all_z = []
    for idx in indices:
        pil_img, _ = dataset[int(idx)]
        views = [meta_encoder(backbone(transform(pil_img).unsqueeze(0).to(device))[1]).squeeze(0)
                 for _ in range(K)]
        all_z.append(torch.stack(views))
    return torch.stack(all_z)  # [N, K, D]

print('Computing augmented z vectors for Option A...')
z_views_a = get_z_for_views(backbone_a, meta_encoder_a, aug_indices,
                             raw_dataset, augment_transform, K_VIEWS, DEVICE)
print('Computing augmented z vectors for Option B...')
z_views_b = get_z_for_views(backbone_b, meta_encoder_b, aug_indices,
                             raw_dataset, augment_transform, K_VIEWS, DEVICE)
print(f'Shape: {tuple(z_views_a.shape)}  (N={N_IMAGES} images × K={K_VIEWS} views × D={z_views_a.shape[-1]} dims)')

In [ ]:
# ── 4b. Within-image z-similarity ─────────────────────────────────────────
def within_image_sims(z_views):
    """Mean pairwise cosine sim across K views for each image. Returns [N]."""
    N, K, _ = z_views.shape
    sims = []
    for i in range(N):
        pair_sims = [
            F.cosine_similarity(z_views[i, j].unsqueeze(0), z_views[i, k].unsqueeze(0)).item()
            for j in range(K) for k in range(j + 1, K)
        ]
        sims.append(float(np.mean(pair_sims)))
    return np.array(sims)

within_a = within_image_sims(z_views_a.cpu())
within_b = within_image_sims(z_views_b.cpu())

print('Within-image z-similarity (different augmented views of the same image):')
print(f'  Option A: mean={within_a.mean():.4f}  std={within_a.std():.4f}')
print(f'  Option B: mean={within_b.mean():.4f}  std={within_b.std():.4f}')
print()
print('Reference — pair z-similarities from Section 3:')
for lbl, zs in [('Option A', z_sims_a), ('Option B', z_sims_b)]:
    print(f'  {lbl}: same-class={zs[same_mask].mean():.4f}  diff-class={zs[diff_mask].mean():.4f}')
print()
print('Expected ordering: within-image > same-class > diff-class')

In [ ]:
# ── 4c. Augmentation invariance plot ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, within, z_sims, label in [
    (axes[0], within_a, z_sims_a, 'Option A (weighted_sum)'),
    (axes[1], within_b, z_sims_b, 'Option B (transformer_cls)'),
]:
    ax.hist(within,             bins=25, alpha=0.7, density=True, color='mediumseagreen',
            label=f'Within-image aug views (mean={within.mean():.3f})')
    ax.hist(z_sims[same_mask],  bins=25, alpha=0.55, density=True, color='steelblue',
            label=f'Same-class pairs (mean={z_sims[same_mask].mean():.3f})')
    ax.hist(z_sims[diff_mask],  bins=25, alpha=0.45, density=True, color='tomato',
            label=f'Diff-class pairs (mean={z_sims[diff_mask].mean():.3f})')
    ax.set_xlabel('z-space cosine similarity')
    ax.set_ylabel('Density')
    ax.set_title(f'{label}\nAugmentation invariance')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('experiments/validation/augmentation_invariance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. z-Space UMAP

Side-by-side: circuit space vs. output (softmax) space, colored by class label.

**What to look for:** Circuit space shows tight class clusters with visible within-class sub-structure. Output space collapses within-class variation onto the softmax simplex. The gap between the two panels is the core CTLS finding made visual.

In [ ]:
# ── 5a. UMAP — Option A ─────────────────────────────────────────────────────────────────────
viz_a = CircuitVisualizer(backbone_a, meta_encoder_a, val_loader, DEVICE)
fig_a = viz_a.plot_umap(title='Option A (weighted_sum)', compare_output=True)
fig_a.savefig('experiments/validation/umap_option_a.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 5b. UMAP — Option B ─────────────────────────────────────────────────────────────────────
viz_b = CircuitVisualizer(backbone_b, meta_encoder_b, val_loader, DEVICE)
fig_b = viz_b.plot_umap(title='Option B (transformer_cls)', compare_output=True)
fig_b.savefig('experiments/validation/umap_option_b.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Appendix: Activation Patching (Supplementary Reference)

Activation patching (replacing x_b's activations at layer l with x_a's and measuring output KL divergence) provides a *causal* circuit similarity measure. However, it has a cascade confound: patching at layer l disrupts all downstream layers simultaneously. For diverse CIFAR-10 pairs on ResNet18, this produces a binary CircuitSim distribution (0 or 1) rather than a continuous one, limiting its use as a quantitative ground truth. The per-layer cosine similarity in Section 2 is the primary ground truth in this notebook.

If you have patching results from a prior run and want a supplementary causal check:

```python
import torch
from scipy.stats import spearmanr

saved_a = torch.load('experiments/validation/patching_results_a.pt', weights_only=False)
sim_scores_a = saved_a['sim_scores']

rho, p = spearmanr(z_sims_a, sim_scores_a)
print(f'Option A: ρ(z-sim, CircuitSim_patching) = {rho:.4f}  p={p:.2e}')
```